In [1]:
import pandas as pd, json, numpy as np
from collections import Counter
from collections import defaultdict


gdf = pd.read_parquet("../data/interim/pois.parquet")
SKIP = {"id","tags","visible","version","timestamp","changeset","geometry","osm_type","lon","lat"}

def parse(t):
    if isinstance(t, dict): return t
    if not isinstance(t, str) or not t: return {}
    try: return json.loads(t)
    except Exception: return {}

def attrs(row):
    d = parse(row.get("tags"))
    for c, v in row.items():
        if c in SKIP or v is None or (isinstance(v, float) and np.isnan(v)) or v == "": continue
        d[c] = v
    return d

gdf["attrs"] = gdf.apply(attrs, axis=1)
N = len(gdf)
cov = Counter(k for d in gdf["attrs"] for k in d)
uniq = defaultdict(set)
for d in gdf["attrs"]:
    for k, v in d.items():
        uniq[k].add(str(v))
ktab = pd.DataFrame({"n": cov}).assign(pct=lambda x: 100*x.n/N).sort_values("n", ascending=False)
ktab["n_unique"] = [len(uniq[k]) for k in ktab.index]
print(len(ktab), "ključeva"); print(ktab.head(40))

988 ključeva
                         n        pct  n_unique
amenity              30226  68.906873       138
name                 23740  54.120597     17615
shop                  9580  21.839736       165
building              7908  18.028041        67
addr:street           7468  17.024963      2102
addr:postcode         6167  14.059045       280
addr:city             6142  14.002052       369
addr:housenumber      5594  12.752764       514
religion              4184   9.538356        12
tourism               4177   9.522398        24
opening_hours         4058   9.251111      1302
phone                 3251   7.411376      3023
website               2903   6.618033      2071
denomination          2799   6.380942        28
operator              2774   6.323948      1135
parking               2426   5.530605        14
access                2140   4.878605        10
name:sr               1874   4.272199      1689
brand                 1799   4.101220       338
fee                   1660 

Ćelija iznad je za svaki red pravila attr kolonu sa svim atributima, koja uključuje merganu tags kolonu sa posebnim tagovima i sve neprazne kolone koje su osm tagovi. Isključene su metapodaci koji su navedeni u listi. Izračunato je da ima čak 988 jedinstvenih ključeva, od kojih amenity pokriva 68 posto svih redova, međutim, samo prvih 8 pokriva preko 10 posto redova tabele.

In [3]:
order = ktab.index.tolist()
total_pairs = ktab.n.sum()
rows = []
for k in [5,10,15,20,30,50,100,len(order)]:
    keep = set(order[:k]) | {"name"}
    kept = ktab.loc[list(keep), "n"].sum()
    empty = (gdf["attrs"].map(lambda d: len(keep & set(d)) == 0)).mean()
    rows.append((k, kept/total_pairs, empty))
print(pd.DataFrame(rows, columns=["k","pairs_retained","rows_with_no_attr"]))

     k  pairs_retained  rows_with_no_attr
0    5        0.458316           0.026673
1   10        0.610836           0.000000
2   15        0.702503           0.000000
3   20        0.759988           0.000000
4   30        0.826225           0.000000
5   50        0.888368           0.000000
6  100        0.943897           0.000000
7  988        1.000000           0.000000

Ovaj cell uzima sve parove, i broji koliko parova od ukupno key-values parova ostaje u attr ako se uzme samo k najčešćih atributa, te paralelno broji koliko redova bi ostalo bez ikakvog atributa. Na 20 najčešćih atributa, a što je već mnogo kolona, vidimo da bi ostali na 25 posto izgubljenih informacija.

In [4]:
descr = [k for k in order if k not in {"ref","url","phone","email","website","source","wikipedia"}
         and ktab.loc[k,"n_unique"] > 1]
covered = np.zeros(N, bool); picked = []
masks = {k: gdf["attrs"].map(lambda d, k=k: k in d).to_numpy() for k in descr[:80]}
for _ in range(15):
    gain = {k: (m & ~covered).sum() for k, m in masks.items() if k not in picked}
    if not gain or max(gain.values()) == 0: break
    k = max(gain, key=gain.get); picked.append(k); covered |= masks[k]
    print(k, gain[k], f"{covered.mean():.1%}")

amenity 30226 68.9%
name 10446 92.7%
shop 1842 96.9%
tourism 1351 100.0%


Pohlepni algoritam računa koliko redova se pokriva minimalnim brojem kolona i koje su kolone. To su ove gore navedene kolone

In [12]:
def txt(d): return " ".join(f"{k} {v}" for k,v in d.items())
lens = gdf["attrs"].map(lambda d: len(txt(d).split()))
print(lens.describe(percentiles=[.5,.9,.99]))

count    43865.000000
mean         9.738448
std         10.518400
min          2.000000
50%          6.000000
90%         22.000000
99%         47.000000
max        423.000000
Name: attrs, dtype: float64


Statistika attrs u smislu ukupnog broja riječi-i key i value su zasebna riječ. 99 posto POI-a ima manje ili jednako 47 riječi. Najduža zabilježena riječ je 423 riječi. Ako value ima više riječi onda i svaka ta pojedinačna riječ ulazi u broj

In [13]:
gdf.shape

(43865, 88)

In [14]:
gdf.columns.tolist()    # imena svih kolona
gdf.info()              # kolone, tipovi i broj nepraznih vrijednosti po koloni
gdf.notna().sum()

<class 'pandas.DataFrame'>
RangeIndex: 43865 entries, 0 to 43864
Data columns (total 88 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   id                43865 non-null  int64  
 1   lon               27017 non-null  float64
 2   lat               27017 non-null  float64
 3   tags              19247 non-null  str    
 4   visible           43865 non-null  bool   
 5   version           43865 non-null  int32  
 6   timestamp         43865 non-null  uint32 
 7   changeset         27139 non-null  float64
 8   addr:city         6142 non-null   str    
 9   addr:country      359 non-null    str    
 10  addr:housenumber  5594 non-null   str    
 11  addr:postcode     6167 non-null   str    
 12  addr:place        292 non-null    str    
 13  addr:street       7468 non-null   str    
 14  email             1161 non-null   str    
 15  name              23740 non-null  str    
 16  opening_hours     4058 non-null   str    
 17  oper

id               43865
lon              27017
lat              27017
tags             19247
visible          43865
                 ...  
coffee               2
security             1
swimming_pool        2
library              1
attrs            43865
Length: 88, dtype: int64

In [16]:
import geopandas as gpd
import pandas as pd
import fiona

fiona.drvsupport.supported_drivers['OSM'] = 'r'

pbf_path="../data/raw/bih.osm.pbf"
layers = fiona.listlayers(pbf_path)

print("Slojevi u sirovom PBF-u:", layers)

gdfs = []
for layer in layers:
    print(f"Učitavam sloj: {layer}...")
    try:
        gdf_layer = gpd.read_file(pbf_path, layer=layer, engine="fiona")
        gdf_layer['pbf_layer'] = layer
        gdfs.append(gdf_layer)
    except Exception as e:
        print(f"Greška na sloju {layer}: {e}")

# 3. Spajanje slojeva
if gdfs:
    gdf_complete_raw = pd.concat(gdfs, ignore_index=True)
    print(f"\nUkupno učitano sirovih zapisa: {len(gdf_complete_raw)}")
    print(f"Ukupno dostupno kolona: {len(gdf_complete_raw.columns)}")

Slojevi u sirovom PBF-u: ['points', 'lines', 'multilinestrings', 'multipolygons', 'other_relations']
Učitavam sloj: points...
Učitavam sloj: lines...
Učitavam sloj: multilinestrings...
Učitavam sloj: multipolygons...
Učitavam sloj: other_relations...

Ukupno učitano sirovih zapisa: 202900
Ukupno dostupno kolona: 35


In [22]:
cols = gdf_complete_raw.columns.tolist()
print(len(cols))
for i in range(0, len(cols), 10):
    print(", ".join(cols[i:i+10]))

35
osm_id, name, barrier, highway, ref, address, is_in, place, man_made, other_tags
geometry, pbf_layer, waterway, aerialway, railway, z_order, type, osm_way_id, aeroway, amenity
admin_level, boundary, building, craft, geological, historic, land_area, landuse, leisure, military
natural, office, shop, sport, tourism


In [2]:
#Analiza sa proširenim filterima

import pandas as pd, json, numpy as np
from collections import Counter
from collections import defaultdict
from pyrosm import OSM

custom_filter = {
    "amenity": [
        "restaurant", "cafe", "fast_food", "bar", "pub", "nightclub",
        "food_court", "ice_cream", "biergarten",
        "bank", "atm", "bureau_de_change", "post_office", "fuel",
        "car_wash", "car_rental", "charging_station",
        "pharmacy", "hospital", "clinic", "doctors", "dentist", "veterinary",
        "school", "university", "college", "kindergarten", "library",
        "place_of_worship", "townhall", "community_centre", "courthouse",
        "cinema", "theatre", "arts_centre", "casino", "events_venue",
        "marketplace",
        "bus_station", "taxi", "police", "fire_station", "social_facility",
    ],
    "shop": True,   
    "tourism": [
        "hotel", "hostel", "motel", "guest_house", "apartment", "camp_site",
        "caravan_site", "chalet", "wilderness_hut", "alpine_hut",
        "museum", "gallery", "attraction", "viewpoint", "theme_park", "zoo",
        "aquarium", "picnic_site",
    ],
    "leisure": [
        "park", "garden", "playground", "dog_park", "nature_reserve",
        "sports_centre", "fitness_centre", "swimming_pool", "stadium",
        "bowling_alley", "water_park", "golf_course", "ice_rink", "marina",
    ],
    "historic": ["castle", "fort", "monument", "memorial", "archaeological_site"],
    "healthcare": ["laboratory", "physiotherapist", "hospice", "alternative"],
    "office": ["company", "government", "estate_agent", "lawyer", "insurance",
               "telecommunication", "accountant", "architect", "ngo",
               "employment_agency", "tax_advisor"],
    "craft": ["tailor", "shoemaker", "carpenter", "electrician", "photographer"],
    "natural": ["peak", "spring", "cave_entrance"],
    "waterway": ["waterfall"],
    "public_transport": ["station"],
    "railway": ["station", "halt"],
    "aeroway": ["aerodrome", "terminal"],
}

extra = ["ruins", "religion", "denomination", "cuisine", "brand", "operator",
         "opening_hours", "name:en", "alt_name", "addr:street",
         "addr:housenumber", "addr:city", "wheelchair", "website", "phone"]


osm = OSM("../data/raw/bih.osm.pbf")
gdf = osm.get_pois(custom_filter=custom_filter,extra_attributes=extra)
print(gdf.shape[0])


44238


In [21]:
cols = gdf.columns.tolist()
print(len(cols))
for i in range(0, len(cols), 10):
    print(", ".join(cols[i:i+10]))

124
id, lon, lat, tags, visible, version, timestamp, changeset, addr:city, addr:country
addr:housenumber, addr:postcode, addr:place, addr:street, email, name, opening_hours, operator, phone, ref
url, website, amenity, atm, bar, bbq, building, building:levels, cafe, car_wash
drinking_water, fast_food, fountain, fuel, gambling, ice_cream, internet_access, landuse, office, parking
post_office, restaurant, school, social_facility, source, start_date, wikipedia, agrarian, alcohol, appliance
bicycle, books, butcher, clothes, craft, fireplace, hairdresser, organic, outdoor, religion
second_hand, shoes, shop, trade, vacuum_cleaner, water, wholesale, attraction, camp_site, guest_house
information, museum, tourism, leisure, outdoor_seating, playground, archaeological_site, historic, memorial, ruins
tomb, architect, company, government, insurance, brewery, natural, sinkhole, waterway, public_transport
station, abandoned, disused, funicular, railway, aeroway, healthcare, denomination, cuisine, bra

In [34]:
def count_uniques(gdf):
    stats = []
    for col in gdf.columns:
        s = gdf[col].dropna()
        vals_count = s.nunique()
        na_count = gdf[col].isna().sum()
        dtype = gdf[col].dtype
        
        stats.append({
            "Column": col,
            "Type": dtype,
            "Uniques": vals_count,
            "Missing": na_count,
            "Sample values": ", ".join(str(v) for v in s.unique()[:3]) 
        })
    
    
    return pd.DataFrame(stats)

pd.set_option('display.max_rows', 150)  # da prikaže svih 124+ reda
pd.set_option('display.max_colwidth', None)
count_uniques(gdf)

,Column,Type,Uniques,Missing,Sample values
0,id,int64,44238,0,"26862607, 26862647, 26862745"
1,lon,float64,30875,13248,"18.0173972, 16.2263889, 17.7511337"
2,lat,float64,30849,13248,"43.3273238, 44.2863889, 42.9705029"
3,tags,str,13874,24776,"{""ele"":""1969"",""name:sr"":""\u0411\u043E\u0442\u0438\u043D""}, {""name:sr"":""\u0411\u0443\u0440\u0441\u0430\u0446\u0438"",""name:sr-Latn"":""Bursaci""}, {""ele"":""955"",""name:hr"":""Crkvina"",""name:sr"":""\u0426\u0440\u043A\u0432\u0438\u043D\u0430""}"
4,visible,bool,1,0,True
5,version,int32,38,0,"7, 3, 9"
6,timestamp,uint32,24265,0,"1783141905, 1639011045, 1782935430"
7,changeset,float64,1,13087,0.0
8,addr:city,str,371,37774,"Sarajevo, Tuzla, Sanski Most"
9,addr:country,str,5,43857,"BA, HR, Bosna i Hercegovina"


In [ ]:
import re

geom_str = gdf['geometry'].astype(str)



0                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                             POINT (18.017397 43.327324)
1                                                             

In [37]:
geom_str = gdf["geometry"].astype(str)


geom_type = geom_str.str.extract(r"^\s*([A-Za-z]+)", expand=False).str.upper()

n_point = (geom_type == "POINT").sum()
print(f"POINT: {n_point} ({n_point/len(gdf):.1%})")

print("\n Other types:")
print(geom_type[geom_type != "POINT"].value_counts(dropna=False))

POINT: 30990 (70.1%)

 Other types:
geometry
POLYGON            13185
MULTIPOLYGON          42
LINESTRING            20
MULTILINESTRING        1
Name: count, dtype: int64


In [38]:
missing_coords = gdf["lon"].isna() | gdf["lat"].isna()
print(f"Rows with NaN in lat or lon: {missing_coords.sum()}")

n_missing_point = (missing_coords & (geom_type == "POINT")).sum()
print(f" POINT: {n_missing_point}")

print("  other geometries:")
print(geom_type[missing_coords & (geom_type != "POINT")].value_counts(dropna=False))

Rows with NaN in lat or lon: 13248
 POINT: 0
  other geometries:
geometry
POLYGON            13185
MULTIPOLYGON          42
LINESTRING            20
MULTILINESTRING        1
Name: count, dtype: int64


In [41]:
def is_missing(s):
    # NaN/None ili potpuno prazna vrijednost ("", "   ", "nan", "None")
    st = s.astype(str).str.strip().str.lower()
    return s.isna() | st.isin(["", "nan", "none", "null"])

lon_missing = is_missing(gdf["lon"])
lat_missing = is_missing(gdf["lat"])

only_lat = lon_missing & ~lat_missing   # ima samo lat
only_lon = ~lon_missing & lat_missing   # ima samo lon
none_ll  = lon_missing & lat_missing    # nema ni jedno

print(f"Samo lat (lon NaN/prazan): {only_lat.sum()}")
print(f"Samo lon (lat NaN/prazan): {only_lon.sum()}")
print(f"Ni lon ni lat:             {none_ll.sum()}")

Samo lat (lon NaN/prazan): 0
Samo lon (lat NaN/prazan): 0
Ni lon ni lat:             13248


In [42]:
# po tipu geometrije: koliko redova nema ni lon ni lat, samo lon, samo lat (NaN ili prazno)
gt = geom_type.fillna("NaN/prazno")
ll_by_type = pd.DataFrame({
    "both_missing": none_ll,
    "only_lon": only_lon,
    "only_lat": only_lat,
}).groupby(gt).sum()

ll_by_type["total_rows"] = gt.value_counts()
ll_by_type.sort_values("total_rows", ascending=False)

,both_missing,only_lon,only_lat,total_rows
geometry,,,,
POINT,0,0,0,30990
POLYGON,13185,0,0,13185
MULTIPOLYGON,42,0,0,42
LINESTRING,20,0,0,20
MULTILINESTRING,1,0,0,1


In [43]:
# duplikati koordinata: redovi koji imaju isti lat I isti lon kao neki drugi red
has_ll = gdf.loc[~lon_missing & ~lat_missing, ["lon", "lat"]]

dup_mask = has_ll.duplicated(subset=["lon", "lat"], keep=False)
dup_groups = has_ll[dup_mask].groupby(["lon", "lat"]).size().sort_values(ascending=False)

print(f"Rows with lat and lon:                       {len(has_ll)}")
print(f"Rows with same coordinates:        {dup_mask.sum()} ({dup_mask.mean():.1%})")
print(f"Locations with multiple rows:              {len(dup_groups)}")
print(f"Duplicate rows: {has_ll.duplicated(subset=['lon', 'lat']).sum()}")

print("\nMost commonly repeated coordinates:")
print(dup_groups.head(10))

Rows with lat and lon:                       30990
Rows with same coordinates:        22 (0.1%)
Locations with multiple rows:              11
Duplicate rows: 11

Most commonly repeated coordinates:
lon        lat      
16.371097  44.319916    2
17.410445  43.369444    2
17.688999  43.186070    2
17.833397  43.240178    2
17.860228  44.606025    2
18.416533  43.858336    2
18.643922  43.510723    2
18.774790  43.506861    2
19.312596  43.884293    2
19.317744  43.881864    2
dtype: int64


In [44]:
import json

def is_empty(v):
    if v is None:
        return True
    if isinstance(v, str):
        return v.strip() == ""
    if isinstance(v, (dict, list, tuple, set)):
        return len(v) == 0
    try:
        return bool(pd.isna(v))
    except (TypeError, ValueError):
        return False

def parse_tags(t):
    if isinstance(t, str):
        try:
            return json.loads(t)
        except ValueError:
            return t
    return None if is_empty(t) else t

def to_jsonable(v):
    # numpy brojeve pretvara u obične Python brojeve, ostalo (npr. geometriju) u string
    if isinstance(v, np.generic):
        return v.item()
    return str(v)

def row_to_json(row):
    d = {
        "id": row["id"],
        "lat": row["lat"],
        "lon": row["lon"],
        "tags": parse_tags(row.get("tags")),
    }
    for col, v in row.items():
        if col in d or col == "json_data" or is_empty(v):
            continue
        d[col] = v
    return json.dumps(d, ensure_ascii=False, default=to_jsonable)

# samo redovi sa dupliciranim koordinatama (dup_mask je indeksiran kao has_ll)
dup_idx = has_ll.index[dup_mask]
dup_rows = gdf.loc[dup_idx].sort_values(["lat", "lon"])

gdf.loc[dup_rows.index, "json_data"] = dup_rows.apply(row_to_json, axis=1)
print(f"Generisano json_data za {len(dup_rows)} redova na {dup_rows.groupby(['lat', 'lon']).ngroups} lokacija")

Generisano json_data za 22 redova na 11 lokacija


In [45]:
pd.set_option("display.max_colwidth", None)
gdf.loc[dup_rows.index, ["id", "lat", "lon", "json_data"]]

,id,lat,lon,json_data
28298,13119870067,43.186070,17.688999,"{""id"": 13119870067, ""lat"": 43.1860697, ""lon"": 17.6889985, ""tags"": null, ""visible"": true, ""version"": 1, ""timestamp"": 1757010006, ""changeset"": 0.0, ""name"": ""Chicago"", ""tourism"": ""guest_house"", ""geometry"": ""POINT (17.6889985 43.1860697)"", ""osm_type"": ""node"", ""match_check"": true}"
28304,13119875111,43.186070,17.688999,"{""id"": 13119875111, ""lat"": 43.1860697, ""lon"": 17.6889985, ""tags"": null, ""visible"": true, ""version"": 1, ""timestamp"": 1757010021, ""changeset"": 0.0, ""name"": ""Chicago"", ""tourism"": ""guest_house"", ""geometry"": ""POINT (17.6889985 43.1860697)"", ""osm_type"": ""node"", ""match_check"": true}"
22361,11073457702,43.240178,17.833397,"{""id"": 11073457702, ""lat"": 43.2401779, ""lon"": 17.8333966, ""tags"": null, ""visible"": true, ""version"": 1, ""timestamp"": 1690554571, ""changeset"": 0.0, ""waterway"": ""waterfall"", ""geometry"": ""POINT (17.8333966 43.2401779)"", ""osm_type"": ""node"", ""match_check"": true}"
24731,11938486036,43.240178,17.833397,"{""id"": 11938486036, ""lat"": 43.2401779, ""lon"": 17.8333966, ""tags"": null, ""visible"": true, ""version"": 1, ""timestamp"": 1716853662, ""changeset"": 0.0, ""waterway"": ""waterfall"", ""geometry"": ""POINT (17.8333966 43.2401779)"", ""osm_type"": ""node"", ""match_check"": true}"
30015,13913344849,43.369444,17.410445,"{""id"": 13913344849, ""lat"": 43.369444, ""lon"": 17.4104451, ""tags"": {""ele"": ""255""}, ""visible"": true, ""version"": 1, ""timestamp"": 1780703383, ""changeset"": 0.0, ""name"": ""Lovreć"", ""natural"": ""peak"", ""geometry"": ""POINT (17.4104451 43.369444)"", ""osm_type"": ""node"", ""match_check"": true}"
30017,13913377997,43.369444,17.410445,"{""id"": 13913377997, ""lat"": 43.369444, ""lon"": 17.4104451, ""tags"": {""ele"": ""255""}, ""visible"": true, ""version"": 1, ""timestamp"": 1780702581, ""changeset"": 0.0, ""name"": ""Lovreć"", ""natural"": ""peak"", ""geometry"": ""POINT (17.4104451 43.369444)"", ""osm_type"": ""node"", ""match_check"": true}"
27917,12999645401,43.506861,18.774790,"{""id"": 12999645401, ""lat"": 43.5068609, ""lon"": 18.7747896, ""tags"": null, ""visible"": true, ""version"": 1, ""timestamp"": 1752515047, ""changeset"": 0.0, ""addr:street"": ""Kralja Petra I"", ""name"": ""ZTR CHS FOČA"", ""opening_hours"": ""Mo-Fr 09:00-21:00; Sa 09:00-18:00"", ""phone"": ""+38765963466"", ""website"": ""http://www.focak.org/"", ""shop"": ""mobile_phone"", ""geometry"": ""POINT (18.7747896 43.5068609)"", ""osm_type"": ""node"", ""match_check"": true}"
27923,12999655701,43.506861,18.774790,"{""id"": 12999655701, ""lat"": 43.5068609, ""lon"": 18.7747896, ""tags"": null, ""visible"": true, ""version"": 1, ""timestamp"": 1752515486, ""changeset"": 0.0, ""addr:street"": ""Kralja Petra I"", ""name"": ""ZTR CHS FOČA"", ""opening_hours"": ""Mo-Fr 09:00-21:00; Sa 09:00-18:00"", ""phone"": ""+38765963466"", ""website"": ""http://www.focak.org/"", ""shop"": ""mobile_phone"", ""geometry"": ""POINT (18.7747896 43.5068609)"", ""osm_type"": ""node"", ""match_check"": true}"
30374,14062210401,43.510723,18.643922,"{""id"": 14062210401, ""lat"": 43.5107233, ""lon"": 18.643922, ""tags"": null, ""visible"": true, ""version"": 1, ""timestamp"": 1785435188, ""changeset"": 0.0, ""amenity"": ""cafe"", ""geometry"": ""POINT (18.643922 43.5107233)"", ""osm_type"": ""node"", ""match_check"": true}"
30375,14062219002,43.510723,18.643922,"{""id"": 14062219002, ""lat"": 43.5107233, ""lon"": 18.643922, ""tags"": null, ""visible"": true, ""version"": 1, ""timestamp"": 1785435621, ""changeset"": 0.0, ""amenity"": ""cafe"", ""geometry"": ""POINT (18.643922 43.5107233)"", ""osm_type"": ""node"", ""match_check"": true}"


In [48]:
# 1) redovi sa istim lat i lon (samo redovi koji imaju obje koordinate)
EXCLUDE = {"id", "timestamp", "json_data"}  # json_data je izvedena kolona, nije dio originalne tabele

t = gdf[gdf["lat"].notna() & gdf["lon"].notna()]
same_coord = t.duplicated(subset=["lat", "lon"], keep=False)
tc = t[same_coord]

# 2) među njima: redovi kod kojih su SVE ostale kolone iste (osim id i timestamp)
#    sve pretvaramo u string da bi se mogle porediti i geometrija/dict vrijednosti; NaN -> "<NA>"
cmp_cols = [c for c in tc.columns if c not in EXCLUDE]
tc_str = tc[cmp_cols].astype(object).where(tc[cmp_cols].notna(), "<NA>").astype(str)
same_all = tc_str.duplicated(keep=False)

print(f"Ukupno redova sa istim koordinatama:                  {same_coord.sum()}")
print(f"  (na {tc.groupby(['lat', 'lon']).ngroups} različitih lokacija)")
print(f"Od toga redova kod kojih su sve ostale kolone iste:   {same_all.sum()}")
print(f"  (u {tc_str[same_all].drop_duplicates().shape[0]} grupa identičnih redova)")
print(f"Redova sa istim koordinatama ali različitim podacima: {(~same_all).sum()}")

Ukupno redova sa istim koordinatama:                  22
  (na 11 različitih lokacija)
Od toga redova kod kojih su sve ostale kolone iste:   14
  (u 7 grupa identičnih redova)
Redova sa istim koordinatama ali različitim podacima: 8


In [49]:
# za svaku lokaciju sa više redova: koje kolone (osim id i timestamp) imaju različite vrijednosti i koje su to vrijednosti
# koristi tc, tc_str i cmp_cols iz prethodne ćelije
diff_rows = []
for (lat, lon), grp in tc_str.join(tc[["id"]]).groupby([tc["lat"], tc["lon"]]):
    diff_cols = [c for c in cmp_cols if grp[c].nunique() > 1]
    print(f"\nLokacija lat={lat}, lon={lon}  | id-evi: {grp['id'].tolist()}")
    if not diff_cols:
        print("  sve kolone iste")
        continue
    for c in diff_cols:
        vals = dict(zip(grp["id"], grp[c]))
        print(f"  {c}:")
        for rid, v in vals.items():
            print(f"      id {rid}: {v}")
        diff_rows.append({"lat": lat, "lon": lon, "column": c, "values": vals})

loc_diff_df = pd.DataFrame(diff_rows)
loc_diff_df


Lokacija lat=43.1860697, lon=17.6889985  | id-evi: [13119870067, 13119875111]
  sve kolone iste

Lokacija lat=43.2401779, lon=17.8333966  | id-evi: [11073457702, 11938486036]
  sve kolone iste

Lokacija lat=43.369444, lon=17.4104451  | id-evi: [13913344849, 13913377997]
  sve kolone iste

Lokacija lat=43.5068609, lon=18.7747896  | id-evi: [12999645401, 12999655701]
  sve kolone iste

Lokacija lat=43.5107233, lon=18.643922  | id-evi: [14062210401, 14062219002]
  sve kolone iste

Lokacija lat=43.8583363, lon=18.4165329  | id-evi: [7909276377, 14288818672]
  tags:
      id 7909276377: {"name:bs":"ARIA","name:hr":"ARIA","name:sr":"\u0410\u0420\u0418\u0410","name:sr-Latn":"ARIA","name:uk":"\u0410\u0420\u0418\u0410","smoking":"separated","toilets":"yes","wikidata":"Q4835047","wikimedia_commons":"Category:ARIA (shopping mall)"}
      id 14288818672: {"check_date":"2026-02-15","note":"Approximate location from public geocoder; field-verify coordinates before relying on them for routing"}
  ve

,lat,lon,column,values
0,43.858336,18.416533,tags,"{7909276377: '{""name:bs"":""ARIA"",""name:hr"":""ARIA"",""name:sr"":""\u0410\u0420\u0418\u0410"",""name:sr-Latn"":""ARIA"",""name:uk"":""\u0410\u0420\u0418\u0410"",""smoking"":""separated"",""toilets"":""yes"",""wikidata"":""Q4835047"",""wikimedia_commons"":""Category:ARIA (shopping mall)""}', 14288818672: '{""check_date"":""2026-02-15"",""note"":""Approximate location from public geocoder; field-verify coordinates before relying on them for routing""}'}"
1,43.858336,18.416533,version,"{7909276377: '5', 14288818672: '2'}"
2,43.858336,18.416533,addr:postcode,"{7909276377: '71000', 14288818672: '<NA>'}"
3,43.858336,18.416533,name,"{7909276377: 'ARIA', 14288818672: 'Telecom Centar Sarajevo City'}"
4,43.858336,18.416533,opening_hours,"{7909276377: 'Mo-Sa 08:00-23:00; Su 08:00-22:00', 14288818672: 'Mo-Fr 08:00-18:00'}"
5,43.858336,18.416533,operator,"{7909276377: '<NA>', 14288818672: 'BH Telecom'}"
6,43.858336,18.416533,phone,"{7909276377: '<NA>', 14288818672: '+38733586939'}"
7,43.858336,18.416533,website,"{7909276377: 'https://www.ariamall.ba/', 14288818672: 'https://www.bhtelecom.ba'}"
8,43.858336,18.416533,office,"{7909276377: '<NA>', 14288818672: 'customer_service'}"
9,43.858336,18.416533,source,"{7909276377: '<NA>', 14288818672: 'BH Telecom official poslovnice list (bhtelecom.ba)'}"


In [50]:
# redovi koji imaju i lat i lon: da li im je geometrija POINT?
with_ll = gdf[gdf["lat"].notna() & gdf["lon"].notna()].copy()
with_ll["geom_type"] = (
    with_ll["geometry"].astype(str)
    .str.extract(r"^\s*([A-Za-z]+)", expand=False)
    .str.upper()
)

not_point = with_ll[with_ll["geom_type"] != "POINT"]

print(f"Redova sa lat i lon:                   {len(with_ll)}")
print(f"  od toga geometrija POINT:            {(with_ll['geom_type'] == 'POINT').sum()}")
print(f"  od toga geometrija NIJE POINT:       {len(not_point)}")
if len(not_point):
    print("\nTipovi geometrije koji nisu POINT:")
    print(not_point["geom_type"].value_counts(dropna=False))

not_point[["id", "lat", "lon", "geom_type", "geometry"]]

Redova sa lat i lon:                   30990
  od toga geometrija POINT:            30990
  od toga geometrija NIJE POINT:       0


,id,lat,lon,geom_type,geometry


In [55]:
gdf[["lat","lon","geometry"]].head(5)

,lat,lon,geometry
0,43.327324,18.017397,POINT (18.0174 43.32732)
1,44.286389,16.226389,POINT (16.22639 44.28639)
2,42.970503,17.751134,POINT (17.75113 42.9705)
3,44.600669,16.526763,POINT (16.52676 44.60067)
4,43.320399,19.085815,POINT (19.08582 43.3204)


In [56]:
# za svaku geometriju koja nije POINT: koliko njenih (lon, lat) parova (tjemena) postoji kao POINT u drugim redovima
import shapely
from collections import defaultdict

gt_all = gdf["geometry"].astype(str).str.extract(r"^\s*([A-Za-z]+)", expand=False).str.upper()
points = gdf[gt_all == "POINT"]
others = gdf[gt_all != "POINT"]

# (lon, lat) -> id-evi POINT redova na toj lokaciji
point_ids = defaultdict(list)
for rid, lon, lat in zip(points["id"], points["lon"], points["lat"]):
    point_ids[(float(lon), float(lat))].append(rid)

res = []
for rid, gtype, geom in zip(others["id"], gt_all[others.index], others["geometry"]):
    coords = shapely.get_coordinates(geom) if geom is not None else []
    pairs = {(float(x), float(y)) for x, y in coords}   # jedinstvena tjemena (zatvoreni poligon ponavlja prvo tjeme)
    matched = [p for p in pairs if p in point_ids]
    res.append({
        "id": rid,
        "geom_type": gtype,
        "n_pairs": len(pairs),
        "n_in_points": len(matched),
        "pct_in_points": 100 * len(matched) / len(pairs) if pairs else 0.0,
        "point_ids": [pid for p in matched for pid in point_ids[p]],
    })

geom_pt_df = pd.DataFrame(res).sort_values("n_in_points", ascending=False)

has_match = geom_pt_df["n_in_points"] > 0
print(f"Geometrija koje nisu POINT:                       {len(geom_pt_df)}")
print(f"  sa barem jednim (lon, lat) parom koji je POINT: {has_match.sum()}")
print(f"Ukupno (lon, lat) parova u tim geometrijama:      {geom_pt_df['n_pairs'].sum()}")
print(f"  od toga postoji kao POINT u drugom redu:        {geom_pt_df['n_in_points'].sum()}")
print("\nPo tipu geometrije:")
print(geom_pt_df.groupby("geom_type").agg(
    geometrija=("id", "size"),
    sa_poklapanjem=("n_in_points", lambda s: (s > 0).sum()),
    parova=("n_pairs", "sum"),
    parova_u_point=("n_in_points", "sum"),
))

geom_pt_df[has_match]

Geometrija koje nisu POINT:                       13248
  sa barem jednim (lon, lat) parom koji je POINT: 51
Ukupno (lon, lat) parova u tim geometrijama:      134850
  od toga postoji kao POINT u drugom redu:        65

Po tipu geometrije:
                 geometrija  sa_poklapanjem  parova  parova_u_point
geom_type                                                          
LINESTRING               20               6     590              13
MULTILINESTRING           1               0      63               0
MULTIPOLYGON             42               1    2404               1
POLYGON               13185              44  131793              51


,id,geom_type,n_pairs,n_in_points,pct_in_points,point_ids
343,126210754,LINESTRING,30,8,26.666667,"[11203747956, 1399701603, 1399701597, 11203747957, 11946084504, 11203747954, 11203747950, 11203747947]"
1000,171468234,POLYGON,15,3,20.000000,"[8968067524, 8968055707, 8968055712]"
2514,242820066,POLYGON,6,2,33.333333,"[3621546846, 3621546847]"
202,91396856,POLYGON,8,2,25.000000,"[4271470501, 4271470492]"
2513,242820059,POLYGON,11,2,18.181818,"[3621546840, 3230579047]"
2337,225371370,POLYGON,20,2,10.000000,"[1840274937, 1840274944]"
13120,10287241,POLYGON,1694,2,0.118064,"[7561695127, 7561695128]"
1563,191304963,POLYGON,7,1,14.285714,[1840686317]
184,82403613,POLYGON,18,1,5.555556,[1749163656]
3441,336705410,POLYGON,14,1,7.142857,[3437709141]


In [58]:
# samo redovi čija geometrija NIJE POINT: broj jedinstvenih vrijednosti i broj NaN po koloni
gt_np = gdf["geometry"].astype(str).str.extract(r"^\s*([A-Za-z]+)", expand=False).str.upper()
non_point = gdf[gt_np != "POINT"]

# astype(str) da bi se mogle brojati i geometrija/dict vrijednosti; NaN se ne broji kao jedinstvena vrijednost
np_counts = pd.DataFrame({
    "n_unique": {c: non_point[c].dropna().astype(str).nunique() for c in non_point.columns},
    "n_nan": non_point.isna().sum(),
})
np_counts["n_not_nan"] = len(non_point) - np_counts["n_nan"]
np_counts["pct_nan"] = (100 * np_counts["n_nan"] / len(non_point)).round(2)

print(f"Redova bez POINT geometrije: {len(non_point)}")
print(f"Kolona koje su potpuno NaN:  {(np_counts['n_not_nan'] == 0).sum()}")
np_counts.sort_values("n_nan")

Redova bez POINT geometrije: 13248
Kolona koje su potpuno NaN:  23


,n_unique,n_nan,n_not_nan,pct_nan
id,13248,0,13248,0.00
timestamp,8919,0,13248,0.00
version,37,0,13248,0.00
visible,1,0,13248,0.00
match_check,1,0,13248,0.00
geometry,13248,0,13248,0.00
osm_type,2,0,13248,0.00
building,68,5262,7986,39.72
name,6245,5789,7459,43.70
amenity,53,5970,7278,45.06


In [59]:
# kolone koje NISU ključevi iz custom_filter niti u listi extra: broj ne-NaN vrijednosti i % NaN (cijeli gdf)
requested = set(custom_filter) | set(extra)
other_cols = [c for c in gdf.columns if c not in requested]

other_stats = pd.DataFrame({
    "n_not_nan": gdf[other_cols].notna().sum(),
    "pct_nan": (100 * gdf[other_cols].isna().mean()).round(2),
}).sort_values("n_not_nan", ascending=False)

print(f"Ukupno redova: {len(gdf)}")
print(f"Kolona van custom_filter/extra: {len(other_cols)} od {len(gdf.columns)}")
other_stats

Ukupno redova: 44238
Kolona van custom_filter/extra: 98 od 126


,n_not_nan,pct_nan
id,44238,0.00
version,44238,0.00
timestamp,44238,0.00
visible,44238,0.00
match_check,44238,0.00
geometry,44238,0.00
osm_type,44238,0.00
changeset,31151,29.58
lat,30990,29.95
lon,30990,29.95


In [5]:
def check_filter_values(df, custom_filter):
    rows = []
    details = {}
    for key, allowed in custom_filter.items():
        if allowed is True:          # "shop": True -> nema ograničenja
            continue
        if key not in df.columns:    # kolona nije ni promovisana u zasebnu
            rows.append({"kolona": key, "status": "nema kolone",
                         "n_van_liste": None, "n_ukupno": 0})
            continue
        allowed_set = set(allowed)
        s = df[key].dropna().astype(str)
        # razbij viševrijednosne ćelije i uparuj svaku vrijednost sa indeksom reda
        exploded = s.str.split(";").explode().str.strip()
        exploded = exploded[exploded != ""]
        mask_bad = ~exploded.isin(allowed_set)
        bad = exploded[mask_bad]
        rows.append({
            "kolona": key,
            "status": "ok" if bad.empty else "ima nepoznatih",
            "n_van_liste": int(mask_bad.sum()),          # pojave, ne redovi
            "n_redova_van": int(df.loc[bad.index.unique()].shape[0]),
            "n_ukupno": int(len(s)),
        })
        if not bad.empty:
            details[key] = bad.value_counts()
    summary = pd.DataFrame(rows).sort_values("n_van_liste",
                                             ascending=False, na_position="last")
    return summary, details

summary, details = check_filter_values(gdf, custom_filter)
print(summary.to_string(index=False))
for key, vc in details.items():
    print(f"\n=== {key}: nepoznate vrijednosti ===")
    print(vc.to_string())

          kolona         status  n_van_liste  n_redova_van  n_ukupno
      healthcare ima nepoznatih         1017          1017      1055
        historic ima nepoznatih          119           118      1644
         amenity ima nepoznatih           51            51     17056
         natural ima nepoznatih           34            23      7886
         tourism ima nepoznatih           21            21      3253
          office ima nepoznatih           14            14       824
public_transport ima nepoznatih           14            14       266
           craft ima nepoznatih           12            12        92
         leisure ima nepoznatih            7             7      3645
        waterway ima nepoznatih            7             7       339
         railway ima nepoznatih            2             2       181
         aeroway             ok            0             0        23

=== amenity: nepoznate vrijednosti ===
amenity
shelter               17
drinking_water         8
ferry